# Problem Set 4: McCall Search with Time-Limited Unemployment Benefits

# Time-Limited Unemployment Benefits

In the baseline McCall model, an unemployed worker receives benefits $c$
in *every* period of unemployment. In practice, U.S. unemployment
insurance (UI) typically lasts only 26 weeks, and then benefits are
exhausted. In addition, UI benefits are typically not available to
workers who have quit their jobs voluntarily. In this problem set you
will extend the McCall model to allow for **time-limited** UI and study
how the imminent exhaustion of benefits changes a worker’s reservation
wage.

Throughout the problem set, use the following baseline parameters
(unless otherwise stated):

$$
\beta = 0.95, \quad S = 50, \quad \bar{w} \in [1, 10]\ (\text{evenly spaced}), \quad p = \text{uniform}, \quad c = 3, \quad c_0 = 0, \quad D = 12
$$

Here $D$ is the maximum number of periods of UI benefits and $c_0$ is
the income an unemployed worker receives once benefits have been
exhausted.

In [1]:
β  = 0.95
S  = 50
w̄  = collect(LinRange(1., 10., S))
p  = ones(S) / S
c  = 3.0      # benefits while eligible
c₀ = 0.0      # consumption after benefits are exhausted
D  = 12       # maximum periods of UI eligibility

12

## Problem 1: Setting Up the Model

Following the lecture, treat employment and unemployment symmetrically:
a worker in state $(k, s)$ holds wage $\bar{w}_s$ and has $k$ periods of
UI eligibility remaining *should they decide not to work*.

To model the fact that you don’t get unemployment benefits if you quit
your job voluntarily, **assume that accepting a job immediately exhausts
all UI eligibility.** A worker who accepts $\bar{w}_s$ today moves into
state $(0, s)$ next period: they cannot return later to claim any of
their original $k$ periods of benefits.

Each period the worker chooses one of two actions:

-   **Accept** (work this period at $\bar{w}_s$): receive $\bar{w}_s$,
    next period transition to state $(0, s)$ — accepting exhausts UI
    eligibility.
-   **Reject** (draw a new offer next period):
    -   If $k \geq 1$: receive benefits $c$, next period draw a new
        offer $\bar{w}_{s'}$ with probabilities $p$ and have $k - 1$
        periods of eligibility left, i.e., transition to state
        $(k-1, s')$.
    -   If $k = 0$ (benefits exhausted): receive $c_0$, next period draw
        a new offer and remain at $k = 0$.

Let $V_{k,s}$ be the value of being in state $(k, s)$, and let
$Q_k = \sum_s p_s V_{k,s}$ be the expected value of a random offer with
$k$ periods of benefits remaining.

**(a)** Write down the Bellman equation for $V_{k,s}$ separately for the
cases $k \geq 1$ and $k = 0$. Each side of the `max` should correspond
to one of the two actions above.

**(b)** Argue that the $k = 0$ case is the same fixed-point problem we
solved in lecture. What is the appropriate value of unemployment
benefits to use in this case?

## Problem 2: Solving the Model

We solve the model in two stages: first the post-exhaustion ($k = 0$)
problem, which by Problem 1(b) is just the standard McCall problem from
lecture; then the $k \geq 1$ problems by **backward induction** in the
benefit counter.

**(a)** Bring in the `iterateBellman` and `solveMcCall` functions from
the lecture and paste them into the cell below. (You can copy them
verbatim — no modifications needed.)

In [1]:
# Your code here: iterateBellman and solveMcCall from lecture


**(b)** Use `solveMcCall` to solve the $k = 0$ problem with the baseline
parameters. Plot $V_{0,s}$ against $\bar{w}_s$ and report the
reservation wage (the lowest wage that the worker accepts when benefits
are exhausted). Make sure to use the correct value of unemployment
benefits in this case.

In [1]:
# Your code here


**(c)** Write a function `iterateBellman_withUI(V₀, Q_prev, β, p, w̄, c)`
that performs one step of the $k \geq 1$ Bellman equation you wrote down
in Problem 1(a), given $V_0$ (already computed in part (b)) and
$Q_{k-1}$. Return a named tuple $(V_k, Q_k, C_k)$ where $V_k$ is the
$S$-vector $\{V_{k,s}\}_s$, $Q_k = p \cdot V_k$, and $C_k$ is the
acceptance policy.

In [1]:
# Your code here


**(d)** Write `solveTimeLimited(β, p, w̄, c, c₀, D)` that does the full
backward induction:

1.  Calls `solveMcCall(β, p, w̄, c₀)` to get $V_0$, $Q_0$, $C_0$ for the
    post-exhaustion problem.
2.  Allocates storage for $V$ as a $(D+1) \times S$ matrix, $Q$ as a
    $(D+1)$-vector, and $C$ as a $(D+1) \times S$ matrix. Use the
    convention that row $k+1$ corresponds to benefit state $k$ (Julia is
    1-indexed).
3.  Loops $k = 1, 2, \ldots, D$, calling
    `iterateBellman_withUI(V₀, Q[k], β, p, w̄, c)` to fill in row $k+1$
    of $V$, $C$, and entry $k+1$ of $Q$.
4.  Returns `(V = V, Q = Q, C = C)`.

In [1]:
# Your code here


**(e)** Solve the model with the baseline parameters. Plot $V_{k,s}$
against $\bar{w}_s$ for $k \in \{0, 2, 4, 8, 12\}$ on the same axes.
Briefly describe what happens to $V_{k,s}$ as $k$ increases.

In [1]:
# Your code here


## Problem 3: Reservation Wages Over the UI Spell

The most economically interesting object in this model is the
**reservation wage as a function of remaining benefits**, $w^*_k$: the
lowest wage at which the worker chooses to accept.

**(a)** Recall from lecture that the reservation wage can be read off
from the acceptance policy as `w̄[findfirst(C .== 1)]`. Using your
solution from Problem 2, compute $w^*_k$ for each
$k \in \{0, 1, \ldots, D\}$ and plot it against $k$. Is the reservation
wage increasing or decreasing in $k$? Give an economic interpretation in
1–2 sentences.

In [1]:
# Your code here


**(b)** **Re-employment around exhaustion.** A famous empirical fact
about UI is that the hazard rate of leaving unemployment *spikes* in the
weeks just before benefits are exhausted. Compute the per-period hazard
rate $h_k = p \cdot C_k$ for each $k$ and plot $h_k$ against $k$. Does
your model reproduce this pattern? Why or why not?

In [1]:
# Your code here


## Problem 4: Comparative Statics

**(a)** **Generosity of benefits ($c$).** Resolve the model for
$c \in \{1, 2, 3, 4, 5\}$, keeping $D = 12$ and $c_0 = 0$. On a single
figure, plot $w^*_k$ versus $k$ for each value of $c$. How does the
level of UI benefits affect the *path* of the reservation wage?

In [1]:
# Your code here


**(b)** **Length of benefits ($D$).** Resolve the model for
$D \in \{4, 12, 26, 52\}$, keeping $c = 3$ and $c_0 = 0$. To compare
across different $D$, plot $w^*_k$ against the number of periods **until
exhaustion** (so $k = 0$ is at the right of the x-axis). What feature of
the reservation-wage path is similar across all $D$, and what feature
differs?

In [1]:
# Your code here


**(c)** **Post-exhaustion safety net ($c_0$).** Fix $c = 3$ and $D = 12$
and resolve the model for $c_0 \in \{0, 1, 2, 3\}$. Plot $w^*_k$ against
$k$ for each $c_0$. Notice that $c_0 = c = 3$ corresponds to the
standard McCall model with permanent benefits. Use this case to *verify
your code*: the reservation wage should be flat in $k$ and equal to the
McCall reservation wage with $c = 3$.

In [1]:
# Your code here


## Problem 5: Simulating Unemployment Spells

Finally, simulate the model to see how time-limited UI affects
unemployment duration.

**(a)** Write a function `simulateSpell(sol, p, w̄, D)` that simulates a
single unemployment spell:

1.  The worker starts at $k = D$ (full benefits) and is unemployed.
2.  Each period, draw a wage offer $\bar{w}_s$ with probabilities $p$.
3.  Use the policy `sol.C` to decide whether to stay (accept) or quit
    (reject). Recall: row $k+1$ stores the policy for benefit state $k$.
4.  If accept, return `(duration, wage, exhausted)` where `exhausted` is
    `true` if the worker reached $k = 0$ before accepting.
5.  If reject, decrement $k$ (but not below $0$).
6.  Cap the simulation at, say, $T_{\max} = 1000$ periods to be safe.

In [1]:
# Your code here


**(b)** Simulate $N = 50{,}000$ spells using the baseline parameters.
Report:

-   the **mean unemployment duration**,
-   the **fraction of workers who exhaust benefits** before finding a
    job,
-   the **mean accepted wage**.

In [1]:
# Your code here


**(c)** Repeat (b) for $D \in \{4, 12, 26\}$. Make a small table
summarizing the three statistics for each $D$. Discuss the trade-off
implied by your results.

> **The UI Trade-off, Revisited**
>
> Time-limited benefits affect re-employment in two opposing ways.
> Longer benefits make workers *more selective early in the spell*
> (raising match quality and the average accepted wage) but *also*
> lengthen unemployment durations. This tension is at the heart of the
> optimal-UI literature: the right level and duration of benefits
> balances consumption smoothing during unemployment against moral
> hazard in search effort.

In [1]:
# Your code here
